In [21]:
# ---------------------------------------------------------------------------
# Unique measured variants with any ClinVar "Uncertain significance" call.
#
# Self-contained: runs on the original integrated-variant-effect dataframe.
# Reproduces Scoreset.splicing_filter (data_utils/dataset.py) inline -- the VEP
# splice-consequence drop and the SpliceAI >= 0.2 drop, both skipped for assays
# that detect splice effects themselves -- since the raw dataframe has not been
# through it yet.
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd

DF = pd.read_csv("/data/ross/assay_calibration/dataframe/integrated_variant_effect_dataset_pp_final.tsv.gz", sep='\t')
GENES_2018 = {"BRCA1", "MSH2", "PTEN", "TP53"}   # clinvar_sig_2018; all others 2025
DATASETS = None               # e.g. {"BRCA1_Findlay_2018", ...} to restrict; None = all
SPLICEAI_THRESHOLD = 0.2      # None disables the SpliceAI drop
VEP_SPLICE_FILTER = True      # False disables the splice-consequence drop

_SPLICE_CONSEQUENCES = {
  "splice region", "splice_site_variant",
  "splice_acceptor_variant", "splice_donor_variant",
}
_SPLICEAI_COLS = ["spliceAI_DS_AG", "spliceAI_DS_AL", "spliceAI_DS_DG", "spliceAI_DS_DL"]

d = DF if DATASETS is None else DF[DF["Dataset"].isin(DATASETS)]
d = d.copy()
print(f"rows in: {len(d):,}  ({d['Dataset'].nunique()} datasets, {d['Gene'].nunique()} genes)")

/tmp/ipykernel_2153642/1545915277.py:13: DtypeWarning: Columns (2,4,6,7,10,11,23,24,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,83,87,94) have mixed types. Specify dtype option on import or set low_memory=False.
  DF = pd.read_csv("/data/ross/assay_calibration/dataframe/integrated_variant_effect_dataset_pp_final.tsv.gz", sep='\t')


rows in: 1,326,110  (90 datasets, 41 genes)


In [22]:

# --- splicing_filter: exempt assays that themselves detect splice effects ---
detects = d.groupby("Dataset")["splice_measure"].transform(lambda s: (s == "Yes").any())

if VEP_SPLICE_FILTER:
  is_splice_csq = d["simplified_consequence"].astype(str).str.lower().isin(_SPLICE_CONSEQUENCES)
  drop_csq = is_splice_csq & ~detects
  print(f"  splice-consequence drop: {int(drop_csq.sum()):,} rows")
  d = d[~drop_csq]
  detects = detects[~drop_csq]

if SPLICEAI_THRESHOLD is not None:
  sai = d[_SPLICEAI_COLS].apply(pd.to_numeric, errors="coerce")
  over = (sai >= SPLICEAI_THRESHOLD).any(axis=1)      # NaN -> False, i.e. kept
  drop_sai = over & ~detects
  print(f"  SpliceAI >= {SPLICEAI_THRESHOLD} drop: {int(drop_sai.sum()):,} rows")
  d = d[~drop_sai]

print(f"rows after splice filtering: {len(d):,}")

# --- per-gene ClinVar release ----------------------------------------------
d["sig_release"] = np.where(d["Gene"].isin(GENES_2018), "2018", "2025")
d["clinvar_sig"] = np.where(
  d["sig_release"] == "2018", d["clinvar_sig_2018"], d["clinvar_sig_2025"],
)
d["is_uncertain"] = d["clinvar_sig"].astype(str).str.strip().str.lower() == "uncertain significance"


  splice-consequence drop: 8,345 rows
  SpliceAI >= 0.2 drop: 6,052 rows
rows after splice filtering: 1,311,713


In [23]:

# --- unique variant key: nucleotide where complete, else amino acid ---------
_nt = ["Gene", "Chrom", "hg38_start", "ref_allele", "alt_allele"]
_aa = ["Gene", "aa_ref", "aa_pos", "aa_alt", "Ensembl Transcript ID"]
has_coord = d[_nt].notna().all(axis=1)
d["variant_key"] = np.where(
  has_coord,
  d[_nt].astype(str).agg("|".join, axis=1),
  "aa|" + d[_aa].astype(str).agg("|".join, axis=1),
)


In [24]:


# --- count: a variant counts if ANY of its measurements is Uncertain --------
uniq = d.groupby("variant_key").agg(
  gene=("Gene", "first"),
  sig_release=("sig_release", "first"),
  is_uncertain=("is_uncertain", "any"),
).reset_index()

In [25]:

print(f"\n{'=' * 70}")
print("UNIQUE MEASURED VARIANTS WITH ANY 'Uncertain significance' CALL")
print('=' * 70)
for rel in ("2025", "2018"):
  sub = uniq[uniq["sig_release"] == rel]
  genes = sorted(sub["gene"].unique())
  print(f"clinvar_sig_{rel}: {int(sub['is_uncertain'].sum()):,} VUS "
        f"of {len(sub):,} unique variants across {len(genes)} genes")
  if rel == "2018":
      print(f"    genes: {', '.join(genes)}")
print(f"TOTAL: {int(uniq['is_uncertain'].sum()):,} VUS of {len(uniq):,} unique "
    f"variants across {uniq['gene'].nunique()} genes")

vus_by_gene = (
  uniq.groupby(["gene", "sig_release"])
  .agg(n_unique_variants=("variant_key", "size"), n_vus=("is_uncertain", "sum"))
  .reset_index().sort_values("n_vus", ascending=False)
)
vus_by_gene


UNIQUE MEASURED VARIANTS WITH ANY 'Uncertain significance' CALL
clinvar_sig_2025: 15,875 VUS of 578,509 unique variants across 37 genes
clinvar_sig_2018: 2,231 VUS of 121,682 unique variants across 4 genes
    genes: BRCA1, MSH2, PTEN, TP53
TOTAL: 18,106 VUS of 700,191 unique variants across 41 genes


,gene,sig_release,n_unique_variants,n_vus
26,PALB2,2025,33046,2196
4,BRCA2,2025,8584,1685
2,BARD1,2025,10911,1673
8,CHEK2,2025,30486,1592
1,BAP1,2025,18108,1189
23,MSH2,2018,53223,1122
19,KCNH2,2025,71141,1012
38,TSC2,2025,27996,910
29,RAD51C,2025,9184,907
30,RAD51D,2025,5410,705
